In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv
import json
import csv
import hashlib
import time

load_dotenv()
MODEL_NAME = "gpt-4o-mini"

In [2]:
client = OpenAI()
api_key = os.environ.get("OPENAI_API_KEY")
if api_key and api_key != "YOUR_KEY_HERE":
    print("API key loaded")
else:
    print("API key not found - check your .env file")

TEMPERATURE = 0.0
MAX_RETRIES = 4
SLEEP_BETWEEN_CALLS_SEC = 0.25

API key loaded


In [3]:
SCHEMA = {
    "type": "object",
    "properties": {
        "category": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of relevant categories for the article."
        },
        "overview": {
            "type": "string",
            "description": "Brief overview extracted from the article"
        },
        "body": {
            "type": "string",
            "description": "Main content of the article"
        },
        "conclusion": {
            "type": "string",
            "description": "Concluding information from the article"
        }
    },
    "required": ["category", "overview", "body", "conclusion"],
    "additionalProperties": False
}

In [4]:
def structure_article(article_text: str) -> dict:
    """Structure a pregnancy article into JSON format."""
    
    messages = [
        {
            "role": "system",
            "content": """
            You are a medical content structuring assistant. 
            Extract and organize the article content into the specified JSON format.
            Do NOT modify, paraphrase, or summarize the original text - preserve it exactly as written.
            Simply organize the existing content into the appropriate sections.
            The category should not be general and must be specific to the article. It is obvious that the article is pregnancy related.
            Hence the category should only be related to the core topic of the article. It should be one to two words max
            """
        },
        {
            "role": "user",
            "content": f"Structure this pregnancy article:\n\n{article_text}"
        }
    ]
    
    resp = client.chat.completions.create(
        model=MODEL_NAME,
        messages=messages,
        temperature=TEMPERATURE,
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "structured_article",
                "strict": True,
                "schema": SCHEMA,
            }
        },
    )
    
    result = json.loads(resp.choices[0].message.content)
    return result